# 01 · Get the DRIAMS data (run once, in one account)
Streams each DRIAMS archive from Dryad and keeps only the 6000-bin spectra and the label files.
The 86 GB DRIAMS-A archive is never stored; the compact copy on Drive is about 2-4 GB.

**Needs a Dryad API account** (see the PDF, step 3). Put the two values into Colab Secrets
(key icon in the left bar): `DRYAD_CLIENT_ID` and `DRYAD_CLIENT_SECRET`, and switch on notebook access.

A GPU is not needed here: `Runtime -> Change runtime type -> CPU` is fine and saves GPU quota.
Keep the tab open until it finishes (DRIAMS-A: roughly 30-90 min, depending on Colab's network).

In [ ]:
# ---- setup: identical in every notebook ----
FOLDER = "AMR_AL"          # name of the shared folder (or its shortcut) in My Drive

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
ROOT = f"/content/drive/MyDrive/{FOLDER}"
if not os.path.isdir(os.path.join(ROOT, "amr_al")):
    raise SystemExit(f"{ROOT}/amr_al not found.\nIn Google Drive open 'Shared with me', right-click {FOLDER} -> "
                     "Organize -> Add shortcut -> My Drive, then run this cell again.")
try:
    import xgboost
    if int(xgboost.__version__.split(".")[0]) < 2:
        raise ImportError
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"], check=True)
os.environ["AMR_AL_ROOT"] = ROOT
sys.path.insert(0, ROOT)
for m in [k for k in sys.modules if k.startswith("amr_al")]:
    del sys.modules[m]                      # always load the latest code from Drive
from amr_al import config, paths, data, models, runner, audit, analysis, driams_fetch
paths.ensure_dirs()
print("shared folder:", ROOT)
print("compute device for XGBoost:", models.device())

## 1 · Dryad token (valid 10 hours)

In [ ]:
try:
    from google.colab import userdata
    cid, secret = userdata.get("DRYAD_CLIENT_ID"), userdata.get("DRYAD_CLIENT_SECRET")
except Exception:
    from getpass import getpass
    cid, secret = getpass("Dryad client id: "), getpass("Dryad client secret: ")
token = driams_fetch.get_token(cid, secret)
assert driams_fetch.test_token(token), "token rejected - check the id/secret"

## 2 · What Dryad has, and a peek inside DRIAMS-A (downloads only a few MB)

In [ ]:
files = driams_fetch.list_files()
for name, f in files.items():
    print(f"{f['size_gb']:7.2f} GB  {name}")
driams_fetch.peek("DRIAMS-A", token, n=40)

Check the peek: you should see paths like `DRIAMS-A/binned_6000/2015/<code>.txt` and `DRIAMS-A/id/2015/2015_clean.csv`
(the order inside the archive may differ). If the folder names look different, stop here and keep the printed list.

## 3 · DRIAMS-A (all four years) - needed for everything

In [ ]:
if "DRIAMS-A" in data.available():
    print("DRIAMS-A already on Drive:", data.available()["DRIAMS-A"], "- delete data/driams_compact/DRIAMS-A to redo")
else:
    man = driams_fetch.fetch_site("DRIAMS-A", token)
driams_fetch.verify_compact("DRIAMS-A")

## 4 · DRIAMS-B, C, D (only for the cross-site experiment E4; B 3.7 GB, C 12 GB, D 43 GB archives)

In [ ]:
GET_OTHER_SITES = False     # set True when you want E4
if GET_OTHER_SITES:
    for site in ["DRIAMS-B", "DRIAMS-C", "DRIAMS-D"]:
        if site in data.available():
            print(site, "already done"); continue
        driams_fetch.fetch_site(site, token)
        driams_fetch.verify_compact(site)

## 5 · Summary

In [ ]:
import json
print(data.available())
for site in data.available():
    man = json.load(open(os.path.join(paths.DATA, site, "MANIFEST.json")))
    for y, info in man["years"].items():
        print(site, y, {k: info.get(k) for k in ["n_matched", "n_id_without_spectrum", "has_acquisition_date"]})
!du -sh "{paths.DATA}"

**Record:** the peek list and the summary output.